In [28]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [21]:
!pip install faiss-cpu
!pip install rank-bm25

In [22]:
%cd /content/gdrive/MyDrive/SPTAR_ADV/SPTAR

/content/gdrive/MyDrive/SPTAR_ADV/SPTAR


In [23]:
import pandas as pd
import numpy as np
import datetime
import os
import re
from sentence_transformers import SentenceTransformer
import faiss
import json
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer
import torch
import time, hashlib, tempfile, shutil
from typing import Dict, Any, List, Optional

## Document to Index

In [24]:
train_path = pd.read_csv('soft_prompt/data/law/prompt_tuning_train_text.csv')
generated_train_path = pd.read_csv('inference_output/law/weak_queries_50_exaone-7b_523_prompt_3.csv', sep="\t")
test_path = pd.read_csv('soft_prompt/data/law/prompt_tuning_test_text.csv')

non_labeled_corpus = []
with open('retrieve/datasets/raw/beir/law/corpus_filtered.jsonl', "r", encoding="utf-8") as f:
    for line in f:
        non_labeled_corpus.append(json.loads(line))

generated_query = []
with open('inference_output/law/weak_queries_50_exaone-7b_523_prompt_3.jsonl', "r", encoding="utf-8") as f:
    for line in f:
        generated_query.append(json.loads(line))

In [25]:
real_query_text = []

for i in range(generated_train_path.shape[0]):
    for j in range(len(generated_query)):
        if str(generated_train_path['query-id'][i]) == str(generated_query[j]['_id']):
            real_query_text.append(generated_query[j]['text'])

generated_train_path['text_x'] = real_query_text

real_corpus_text = []

for i in range(generated_train_path.shape[0]):
    for j in range(len(non_labeled_corpus)):
        if str(generated_train_path['corpus-id'][i]) == str(non_labeled_corpus[j]['_id']):
            real_corpus_text.append(non_labeled_corpus[j]['text'])

generated_train_path['text_y'] = real_corpus_text

generated_train_path.head(2)

,query-id,corpus-id,score,text_x,text_y
0,5000001,2613,1,하자와의 분쟁에 대한 조정이 이루어지지 않아 강제경매를 진행한 경우 그 부동산을 취...,가등기담보 등에 관한 법률 15조 제15조(담보가등기권리의 소멸) 담보가등기를 마친...
1,5000002,2614,1,서울에서 자영업자로 음식점을 운영하고 있습니다 저녁시간에는 손님이 별로 없어 종업원...,가사근로자의 고용개선 등에 관한 법률 11조 제3장 가사서비스의 제공 제11조(가사...


In [26]:
new_train_path = pd.concat([generated_train_path, train_path])
all_corpus = pd.concat([new_train_path, test_path]).drop_duplicates()
all_corpus.shape

(3643, 5)

In [10]:
class PrebuiltIndexNotFound(Exception): ...
class ArtifactMissing(Exception): ...

# util
def _safe_makedirs(p: str): os.makedirs(p, exist_ok=True)

def _atomic_write_bytes(dst: str, data: bytes):
    d = os.path.dirname(dst); _safe_makedirs(d)
    fd, tmp = tempfile.mkstemp(dir=d)
    try:
        with os.fdopen(fd, "wb", buffering=0) as f:
            f.write(data); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, dst)
    except Exception:
        try: os.remove(tmp)
        finally: raise

def _save_json(path: str, obj: Any):
    _atomic_write_bytes(path, json.dumps(obj, ensure_ascii=False, indent=2).encode("utf-8"))

def _load_json(path: str) -> Any:
    if not os.path.exists(path): raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8") as f: return json.load(f)

def _save_numpy_atomic(path: str, arr: np.ndarray):
    d = os.path.dirname(path); _safe_makedirs(d)
    fd, tmp = tempfile.mkstemp(dir=d)
    try:
        with os.fdopen(fd, "wb", buffering=0) as f:
            np.save(f, arr); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, path)
    except Exception:
        try: os.remove(tmp)
        finally: raise

def _load_numpy(path: str) -> np.ndarray:
    if not os.path.exists(path): raise FileNotFoundError(path)
    return np.load(path)

def _save_faiss(path: str, index: faiss.Index):
    _safe_makedirs(os.path.dirname(path)); faiss.write_index(index, path)

def _load_faiss(path: str) -> faiss.Index:
    if not os.path.exists(path): raise FileNotFoundError(path)
    return faiss.read_index(path)

# 코퍼스 키: 문서+instruction만 반영
def _hash_corpus(docs: List[str], instruction: bool) -> str:
    h = hashlib.sha256()
    h.update(b"inst1" if instruction else b"inst0")
    for d in docs:
        h.update(b"\x1e"); h.update(d.encode("utf-8"))
    return h.hexdigest()[:16]

def _paths_corpus(store_dir: str, corpus_key: str) -> Dict[str, str]:
    root = os.path.join(store_dir, corpus_key)
    return {
        "root": root,
        "corpus": os.path.join(root, "corpus.json"),
        "corpus_config": os.path.join(root, "corpus_config.json"),
        "bm25_tokens": os.path.join(root, "bm25_tokens.json"),
        "variants_dir": os.path.join(root, "variants"),
    }

def _paths_variant(store_dir: str, corpus_key: str, variant: str) -> Dict[str, str]:
    vd = os.path.join(store_dir, corpus_key, "variants", variant)
    return {
        "dir": vd,
        "config": os.path.join(vd, "config.json"),
        "dense_npy": os.path.join(vd, "dense_emb.npy"),
        "faiss_index": os.path.join(vd, "faiss.index"),
    }

# 임베딩 저장
def build_and_save_index(
    *,
    docs: List[str],
    instruction: bool,
    variant: str,                 # ex) "bge-m3" | "sbert" | "other"
    embed_model_name: str,        # ex) "BAAI/bge-m3", "sentence-transformers/all-MiniLM-L6-v2", .
    method: str,                  # "dense" | "faiss" | "bm25"
    store_dir: str = "embedded_docs",
    corpus_key: Optional[str] = None,   # 미지정 시 자동 생성
    overwrite_variant: bool = False,    # 같은 variant 덮어쓸지
    build_bm25_once: bool = False,      # BM25 토큰도 함께 만들고 싶으면 True
) -> str:
    # instruction 프리픽스 적용
    _docs = [f"Document: {d}" for d in docs] if instruction else list(docs)
    corpus_key = corpus_key or _hash_corpus(_docs, instruction)

    PC = _paths_corpus(store_dir, corpus_key)
    PV = _paths_variant(store_dir, corpus_key, variant)

    # 코퍼스 메타/문서 저장(없으면)
    _safe_makedirs(PC["root"])
    if not os.path.exists(PC["corpus"]):
        _save_json(PC["corpus"], _docs)
        _save_json(PC["corpus_config"], {
            "instruction": instruction,
            "created_at": int(time.time()),
            "version": 1
        })

    # BM25 토큰 생성(옵션, 중복 방지)
    if build_bm25_once and not os.path.exists(PC["bm25_tokens"]):
        tok = AutoTokenizer.from_pretrained(embed_model_name)
        tokenized = [tok.tokenize(d) for d in _docs]
        _save_json(PC["bm25_tokens"], tokenized)

    # 변종 생성
    if (os.path.exists(PV["config"]) or os.path.exists(PV["dense_npy"]) or os.path.exists(PV["faiss_index"])) and not overwrite_variant:
        return corpus_key  # 이미 있음

    _safe_makedirs(PV["dir"])

    if method == "bm25":
        # 변종에 별도 파일은 없음(코퍼스 공용 bm25_tokens 사용)
        _save_json(PV["config"], {
            "variant": variant, "embed_model_name": embed_model_name,
            "method": "bm25", "created_at": int(time.time())
        })
        return corpus_key

    if method not in {"dense", "faiss"}:
        raise ValueError("method must be one of {'bm25','dense','faiss'}")

    model = SentenceTransformer(embed_model_name)
    emb = model.encode(_docs, convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    _save_numpy_atomic(PV["dense_npy"], emb)

    if method == "faiss":
        dim = emb.shape[1]
        index = faiss.IndexFlatIP(dim)
        index.add(emb)
        _save_faiss(PV["faiss_index"], index)

    _save_json(PV["config"], {
        "variant": variant,
        "embed_model_name": embed_model_name,
        "method": method,
        "dim": int(emb.shape[1]) if method in {"dense","faiss"} else None,
        "created_at": int(time.time())
    })
    return corpus_key

# 로드
def load_index(
    *,
    store_dir: str,
    corpus_key: str,
    variant: str,
    use_gpu_for_faiss: bool = True
) -> Dict[str, Any]:
    PC = _paths_corpus(store_dir, corpus_key)
    PV = _paths_variant(store_dir, corpus_key, variant)
    if not os.path.exists(PC["corpus"]):
        raise PrebuiltIndexNotFound(f"Corpus not found: {PC['root']}")

    docs = _load_json(PC["corpus"])
    if not os.path.exists(PV["config"]):
        raise PrebuiltIndexNotFound(f"Variant not found: {PV['dir']}")

    vcfg = _load_json(PV["config"])
    method = vcfg["method"]

    if method == "bm25":
        if not os.path.exists(PC["bm25_tokens"]):
            raise ArtifactMissing("bm25_tokens.json missing for corpus")
        tokens = _load_json(PC["bm25_tokens"])
        tokenizer = AutoTokenizer.from_pretrained(vcfg["embed_model_name"])
        bm25 = BM25Okapi(tokens)
        return {"method": "bm25", "docs": docs, "bm25": bm25, "tokenizer": tokenizer}

    if method == "dense":
        if not os.path.exists(PV["dense_npy"]):
            raise ArtifactMissing("dense_emb.npy missing for variant")
        emb = _load_numpy(PV["dense_npy"]).astype("float32")
        model = SentenceTransformer(vcfg["embed_model_name"])
        return {"method": "dense", "docs": docs, "docs_embeddings": emb, "dense_model": model}

    if method == "faiss":
        if not (os.path.exists(PV["dense_npy"]) and os.path.exists(PV["faiss_index"])):
            raise ArtifactMissing("faiss.index or dense_emb.npy missing for variant")
        emb = _load_numpy(PV["dense_npy"]).astype("float32")
        index = _load_faiss(PV["faiss_index"])  # CPU
        if use_gpu_for_faiss and torch.cuda.is_available():
            res = faiss.StandardGpuResources()
            index = faiss.index_cpu_to_gpu(res, 0, index)
        model = SentenceTransformer(vcfg["embed_model_name"])
        return {"method": "faiss", "docs": docs, "docs_embeddings": emb, "dense_model": model, "faiss_index": index}

    raise ValueError(f"unknown method in variant config: {method}")

# 검색
def search(query: str, k: int, handle: Dict[str, Any]):
    m = handle["method"]; docs = handle["docs"]
    if m == "bm25":
        tok = handle["tokenizer"]; bm25 = handle["bm25"]
        q_tokens = tok.tokenize(query)
        scores = bm25.get_scores(q_tokens)
        idx = np.argsort(scores)[::-1][:k]
        return [(docs[i], float(scores[i])) for i in idx]
    if m == "dense":
        model = handle["dense_model"]; emb = handle["docs_embeddings"]
        qv = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0].astype("float32")
        sims = emb @ qv
        idx = np.argsort(sims)[::-1][:k]
        return [(docs[i], float(sims[i])) for i in idx]
    if m == "faiss":
        model = handle["dense_model"]; index = handle["faiss_index"]
        qv = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
        D, I = index.search(qv, k)
        return [(docs[int(i)], float(D[0, j])) for j, i in enumerate(I[0])]
    raise ValueError("unknown method in handle")


In [11]:
variants      = ["bge-m3", "ko-legal-sbert", "qwen3-embedding"]
model_list    = ["upskyy/bge-m3-korean", "woong0322/ko-legal-sbert-finetuned"]#, "Day1Kim/Qwen3-Embedding-0.6B-Korean"]
method_list   = ["bm25", "dense", "faiss"]
corpus_key    = "corpus-all"
# corpus_key = "law-v1"

# 확인: 아래 한 줄이 위를 즉시 덮어씌웁니다. 의도된 건지 점검하세요.
# docs = list(generated_train_path['text_y'])
docs = list(all_corpus['text_y'])  # 최종적으로 이걸 사용하게 됨

In [ ]:

base_variant = "bge-m3"
for method in method_list:
    # (모델 × 메서드) 조합으로 variant 분기
    variant_name = f"{base_variant}@{method}"

    build_bm25 = (method == "bm25")  # bm25 토큰은 코퍼스당 한 번만 생성, 중복 생성해도 내부에서 한 번만 기록됨
    build_and_save_index(
        docs=docs,
        instruction=True,
        variant=variant_name,
        embed_model_name="upskyy/bge-m3-korean",
        method=method,                  # "bm25" | "dense" | "faiss"
        store_dir="embedded_docs",
        corpus_key=corpus_key,
        overwrite_variant=False,        # 같은 variant가 이미 있으면 건너뜀 (variant를 메서드별로 달리 주니 충돌 없음)
        build_bm25_once=build_bm25
    )


Token indices sequence length is longer than the specified maximum sequence length for this model (1057 > 512). Running this sequence through the model will result in indexing errors


In [ ]:

base_variant = "ko-legal-sbert"
for method in method_list:
    # (모델 × 메서드) 조합으로 variant 분기
    variant_name = f"{base_variant}@{method}"

    build_bm25 = (method == "bm25")  # bm25 토큰은 코퍼스당 한 번만 생성, 중복 생성해도 내부에서 한 번만 기록됨
    build_and_save_index(
        docs=docs,
        instruction=False,
        variant=variant_name,
        embed_model_name= "woong0322/ko-legal-sbert-finetuned",
        method=method,                  # "bm25" | "dense" | "faiss"
        store_dir="embedded_docs",
        corpus_key=corpus_key,
        overwrite_variant=False,        # 같은 variant가 이미 있으면 건너뜀 (variant를 메서드별로 달리 주니 충돌 없음)
        build_bm25_once=build_bm25
    )


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
# OOM 문제로 로드 불가능

# base_variant = "qwen3-embedding"
# for method in method_list:

#     # (모델 × 메서드) 조합으로 variant 분기
#     variant_name = f"{base_variant}@{method}"

#     build_bm25 = (method == "bm25")  # bm25 토큰은 코퍼스당 한 번만 생성, 중복 생성해도 내부에서 한 번만 기록됨
#     build_and_save_index(
#         docs=docs,
#         instruction=False,
#         variant=variant_name,
#         embed_model_name="Day1Kim/Qwen3-Embedding-0.6B-Korean",
#         method=method,                  # "bm25" | "dense" | "faiss"
#         store_dir="embedded_docs",
#         corpus_key=corpus_key,
#         overwrite_variant=False,        # 같은 variant가 이미 있으면 건너뜀 (variant를 메서드별로 달리 주니 충돌 없음)
#         build_bm25_once=build_bm25
#     )


## Train Retriever

In [ ]:
'''
실험 목적
### 생성된 weak query가 실제로 연관된 문서를 잘 검색하는가?를 알기 위함
### 생성 전(train_path)의 데이터로부터 검색기를 학습시킨다
### 생성된 weak query data(train_path + generated_train_path) 로부터 검색기를 학습시킨다

### 둘의 결과를 평가한다. (by test_path)
### 학습 평가는 정확도로 총 2개를 비교한다. top=1으로 golden docs를 가져왔는지 & top=15 중 golden docs가 포함되어있는지

실험 세팅
- embedding model : "upskyy/bge-m3-korean", "woong0322/ko-legal-sbert-finetuned"
- method : "bm25", "dense", "faiss"
- corpus_key : "corpus-all"
  -> 해당 파일 안에 : bge@bm25, bge@dense, bge@faiss, sbert@bm25, sbert@dense, sbert@faiss 등의 파일이 존재함 -> 이 6개의 조합에 대한 실험 진행
- corpus for train : train_path['text_y'] 와 train_path['text_y'] + generated_train_path['text_y']로 비교

'''

In [14]:
# 예: BGE + FAISS

def search_inference(query, corpus_key="corpus-all", variant="bge-m3@faiss", use_gpu_for_faiss=False):
    index_setting = load_index(store_dir="embedded_docs", corpus_key=corpus_key, variant=variant, use_gpu_for_faiss=use_gpu_for_faiss)
    if 'bge' in variant:
      query = f"Query: {query}"
    res = search(query, k=15, handle=index_setting)
    return res


In [17]:
# -*- coding: utf-8 -*-
from typing import List, Tuple, Dict, Optional
import re
import pandas as pd


def _normalize(t: str) -> str:
    # 공백/개행 제거 + 일부 기호 정규화 (필요시 규칙 추가)
    t = t.replace('\u00B7','·').replace('ㆍ','·')
    t = re.sub(r'\s+', '', t)
    t = re.sub(r'[“”„‟″]', '"', t)
    t = re.sub(r"[’‘′`']", "'", t)
    t = t.replace('–','-').replace('—','-')
    return t

def head_key(t: str, n: int = 100) -> str:
    return _normalize(t)[-n:]

def build_head100_index(
    corpus_texts: List[str],
    corpus_ids: List[int],
    n: int = 100
) -> Dict[str, List[int]]:
    assert len(corpus_texts) == len(corpus_ids)
    idx: Dict[str, List[int]] = {}
    for cid, txt in zip(corpus_ids, corpus_texts):
        k = head_key(txt, n=n)
        idx.setdefault(k, []).append(int(cid))
    return idx


def map_text_to_corpus_id(
    text: str,
    index: Dict[str, List[int]],
    n: int = 100,
    on_collision: str = "first",  # {"first","none","error"}
) -> Optional[int]:
    k = head_key(text, n=n)
    hit = index.get(k)
    if not hit:
        return None
    if len(hit) == 1:
        return hit[0]
    if on_collision == "first":
        return hit[0]
    elif on_collision == "none":
        return None
    elif on_collision == "error":
        raise ValueError(f"Ambiguous head-{n} match: {hit}")
    else:
        raise ValueError("on_collision must be one of {'first','none','error'}")

def attach_corpus_ids_to_results(
    results: List[Tuple[str, float]],
    index: Dict[str, List[int]],
    n: int = 100,
    on_collision: str = "first"
) -> List[Tuple[Optional[int], str, float]]:
    """
    returns: [(corpus_id_or_None, text, score), ...]
    """
    out = []
    for text, score in results:
        cid = map_text_to_corpus_id(text, index, n=n, on_collision=on_collision)
        out.append((cid, text, score))
    return out


docs = list(all_corpus['text_y'])
corpus_ids = list(all_corpus['corpus-id'])  # 하이픈 컬럼명 주의

# 인덱스 빌드 (앞 100자)
idx100 = build_head100_index(docs, corpus_ids, n=100)

query = test_path['text_x'][0]


results = search_inference(query, corpus_key="corpus-all", variant="bge-m3@faiss", use_gpu_for_faiss=False)

# corpus-id 부착
mapped = attach_corpus_ids_to_results(results, idx100, n=100, on_collision="first")

# 필요하면 DataFrame으로 보기 좋게
mapped_df = pd.DataFrame(mapped, columns=["corpus-id", "text", "score"])
#

In [18]:
mapped_df # 이렇게 부착한 corpus-id와 실제 data 속 corpus-id를 비교하여 정확도 추출

,corpus-id,text,score
0,2617,Document: 가사근로자의 고용개선 등에 관한 법률 15조 제15조(최소근로시간...,0.622086
1,2617,Document: 가사근로자의 고용개선 등에 관한 법률 15조 제15조(최소근로시간...,0.617974
2,2617,Document: 가사근로자의 고용개선 등에 관한 법률 15조 제15조(최소근로시간...,0.617974
3,3082,Document: 남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률 시행령 10조 제1...,0.586286
4,3082,Document: 남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률 시행령 10조 제1...,0.586286
5,3082,Document: 남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률 시행령 10조 제1...,0.584307
6,2937,Document: 국가공무원 복무규정 10조 제10조(근무시간 등의 변경)① 중앙행...,0.547276
7,2380,Document: 국가공무원 복무규정 10조 제10조(근무시간 등의 변경) ① 중앙...,0.539266
8,2380,Document: 국가공무원 복무규정 10조 제10조(근무시간 등의 변경) ① 중앙...,0.539266
9,2380,Document: 국가공무원 복무규정 10조 제10조(근무시간 등의 변경) ① 중앙...,0.539266


In [ ]:
## 유하가 할 일
## 학습 전에 학습하지 않은 retriever 들의 성능을 비교함 -> 총 6개
### 평가한다. (by test_path)
### 학습 평가는 정확도로 총 2개를 비교한다. top=1으로 golden docs를 가져왔는지 & top=15 중 golden docs가 포함되어있는지
'''
실험 세팅
- embedding model : "upskyy/bge-m3-korean", "woong0322/ko-legal-sbert-finetuned"
- method : "bm25", "dense", "faiss"
- corpus_key : "corpus-all"
  -> 해당 파일 안에 : bge@bm25, bge@dense, bge@faiss, sbert@bm25, sbert@dense, sbert@faiss 등의 파일이 존재함 -> 이 6개의 조합에 대한 실험 진행
- corpus for train : train_path['text_y'] 와 train_path['text_y'] + generated_train_path['text_y']로 비교


In [ ]:
### 여기에 코드 작성

In [1]:
# 여기부터는 안 봐도 됨!!

train_pairs = []
dev_pairs = []

for i in range(train_path.shape[0]):
  train_pairs.append((train_path['text_x'][i], train_path['text_y'][i], train_path['score'][i]))

for i in range(test_path.shape[0]):
  dev_pairs.append((test_path['text_x'][i], test_path['text_y'][i], test_path['score'][i]))

train_pairs

NameError: name 'train_path' is not defined

In [ ]:
import math
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
import os

os.makedirs('train_retriever_result')

# 2) 모델 로드
model_name = "woong0322/ko-legal-sbert-finetuned"
model = SentenceTransformer(model_name)

# 3) DataLoader
train_examples = [InputExample(texts=[a, b], label=int(y)) for a, b, y in train_pairs]
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)

# 4) Loss: Contrastive (라벨 0/1)
#   distance_metric 기본은 Cosine로 동작 (내부적으로 1 - cos_sim 사용)
#   margin은 보통 0.5~0.7 사이에서 시작해 튜닝
train_loss = losses.ContrastiveLoss(model, margin=0.5)

# 5) 평가 지표: BinaryClassificationEvaluator (0/1 라벨)
#   sentence-transformers에 포함되어 있음
dev_sents1 = [a for a, _, _ in dev_pairs]
dev_sents2 = [b for _, b, _ in dev_pairs]
dev_labels = [int(y) for _, _, y in dev_pairs]
evaluator = evaluation.BinaryClassificationEvaluator(
    dev_sents1, dev_sents2, dev_labels, name="dev-bin"
)

# 6) 학습
num_epochs = 3
warmup_steps = math.ceil(len(train_dataloader) * num_epochs * 0.1)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,            # 없애도 됨
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    evaluation_steps=200,           # 데이터 크기에 맞게 조정
    output_path="train_retriever_result/ko-legal-sbert-contrastive-01"
)

#
from sentence_transformers import util
m = SentenceTransformer("train_retriever_result/ko-legal-sbert-contrastive-01")
q =test_path['text_x'][0]
c = test_path['text_y'][0]
sim = util.cos_sim(m.encode(q), m.encode(c)).item()
print("cosine:", sim)